In [ ]:
%pip install -U torch torchvision transformers datasets evaluate accelerate scikit-learn emoji peft bitsandbytes

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

import re

In [ ]:
# Load and preprocess the dataset
temp = load_dataset("BrianLeung/sentiment140-csv", encoding="ISO-8859-1")["train"]
temp = temp.remove_columns(["ID", "Date", "Flag", "User"])
# Remove unused columns
temp = temp.filter(lambda x: x["Label"] in [0, 4])  # Keep only binary sentiment labels

# Relabel function: 0 -> 0 (negative), 4 -> 1 (positive)
def relabel(batch):
    labels = batch["Label"]
    new_labels = [0 if l == 0 else 1 if l == 4 else None for l in labels]
    if any(l is None for l in new_labels):
        raise ValueError("Unexpected label in batch")
    return {"Label": new_labels}

# Replace usernames and URLs with placeholders
def replace(text):
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'http\S+|www\.\S+', 'HTTPURL', text)
    return text

# Clean text in batch
def clean_text(batch):
    return {"Text": [replace(t) for t in batch["Text"]] }

temp = temp.map(relabel, batched=True)
temp = temp.map(clean_text, batched=True)
temp = temp.train_test_split(test_size=0.1, seed=1)
test_valid = temp["test"].train_test_split(test_size=0.5, seed=1)

# Organize splits into a dictionary
dataset_dict = {
    "train": temp["train"].shuffle(seed=1),
    "test": test_valid["test"],
    "validation": test_valid["train"],
}

dataset_dict

In [ ]:
# Load tokenizer and model
model_path = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Define label mappings
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=2, id2label=id2label, label2id=label2id)

In [ ]:
# Tokenization function (fixed max_length for LoRA)
def preprocess_function(x):
    return tokenizer(x["Text"], truncation=True, padding=False, max_length=128)

# Tokenize all splits and rename label column
tokenized = {}
for split, ds in dataset_dict.items():
    ds = ds.rename_column("Label", "labels")
    tokenized[split] = ds.map(preprocess_function, batched=True)

In [ ]:
# Print names of modules to be targeted for LoRA
for n, m in model.named_modules():
    if any(k in n.lower() for k in ("q_proj","v_proj","query","key","value","dense","output","proj","attn")):
        print(n)

# Prepare model for k-bit (quantized) training
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig, get_peft_model
# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
import os

OUTPUT_DIR = "/content/drive/MyDrive/outputs/lora-bertweet"

RESUME_CHECKPOINT = ""
# RESUME_CHECKPOINT = "/content/drive/MyDrive/outputs/lora-bertweet/checkpoint-45000"

resume_from_checkpoint = (
    RESUME_CHECKPOINT if (RESUME_CHECKPOINT and os.path.isdir(RESUME_CHECKPOINT)) else None
 )

if resume_from_checkpoint:
    print(f"Resuming from checkpoint: {resume_from_checkpoint}")
else:
    print("No checkpoint found (or not set). Training from scratch.")

In [ ]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from transformers.utils.notebook import NotebookProgressCallback
import evaluate, numpy as np

_output_dir = OUTPUT_DIR

_resume_from_checkpoint = resume_from_checkpoint

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load evaluation metrics
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

# Compute metrics for evaluation
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

# Training arguments
training_args = TrainingArguments(
    output_dir=_output_dir,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    report_to="none",   # Colab Error
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
 )

# Colab Error
trainer.remove_callback(NotebookProgressCallback)

# Train and evaluate
trainer.train(resume_from_checkpoint=_resume_from_checkpoint)
results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results["eval_accuracy"])
print("Test F1:", results["eval_f1"])

In [ ]:
# Save the trained model
model.save_pretrained('/content/drive/MyDrive/outputs/lora-bertweet/final')